# NeuroDecode-LaBraM — Phase 3 Validation Rerun (Reproducibility Gate)

This version addresses the issues found in the **first real Kaggle run**:

1. Riemannian logistic regression now standardizes tangent features **on training data only**, uses a higher iteration limit, and **fails** if convergence is not reached. This is a **revised baseline**, so new validation scores should be compared with the archived first-run results with that caveat.
2. EEGNet and pretrained LaBraM save their selected **best-epoch weights**, record SHA-256, reload those weights, and check that all 2,436 validation probabilities match before declaring success. Checkpoints are stored in the Kaggle working output tree; they are deliberately excluded from the downloadable results ZIP.
3. All baseline evaluations remain **VALIDATION ONLY**. The frozen 15-subject Test set is never evaluated.

**Requires:** previously validated 104-subject HDF5, CUDA recommended, Internet enabled. This run does not overwrite the earlier Phase 3 result archive. Model seeds and participant split remain unchanged. The selected checkpoint models are not claimed to be independent test benchmarks.


In [ ]:
import os, sys, subprocess, pathlib, json, glob, zipfile, hashlib, datetime
from IPython.display import display
print('Python',sys.version.split()[0]); print('Kaggle directory', os.getcwd())
ROOT = pathlib.Path('/kaggle/working' if pathlib.Path('/kaggle').exists() else '/mnt/data')
WORKSPACE = ROOT / 'neurodecode_phase3_checkout'
OUTPUT = ROOT / 'NeuroDecode_Phase3_Validation_Rerun'
RESULT_ZIP = ROOT / 'NeuroDecode_Phase3_Validation_Rerun.zip'
# Optional: supply a single explicit absolute path below to override search.
CACHE_OVERRIDE = ''
# Can disable an expensive model intentionally but then Phase 3 remains PARTIAL.
MODELS = ['dummy', 'riemann', 'eegnet', 'labram_probe']
SEED = 42
EPOCHS = 30
PATIENCE = 6
BATCH_SIZE = 32
BOOTSTRAPS = 250
print('Models selected:', MODELS)

### 1 · Retrieve **pinned and audited** repository commit

Do not silently pull newer commits into this benchmark. Phase-2-preprocessing source must match the frozen contract.

In [ ]:
if not WORKSPACE.exists():
    subprocess.run(['git','clone','https://github.com/mohamad679/labram-mi-project.git',str(WORKSPACE)],check=True)
subprocess.run(['git','-C',str(WORKSPACE),'checkout','--detach','b3af6247d027cd1a3dd33915368bad03f9bf220c'],check=True)
revision=subprocess.check_output(['git','-C',str(WORKSPACE),'rev-parse','HEAD'],text=True).strip()
assert revision == 'b3af6247d027cd1a3dd33915368bad03f9bf220c'
os.chdir(WORKSPACE)
print('Frozen code revision:',revision)

### 2 · Install checked model libraries

Includes pyRiemann for the spatial-covariance baseline and Braindecode 1.5.1 for EEGNet/LaBraM. Kaggle supplies PyTorch; this cell does not intentionally reinstall the CUDA runtime.

In [ ]:
pkgs=['braindecode[hub]==1.5.1','moabb==1.7.2','mne==1.13.2',
      'pyriemann==0.12','h5py>=3.10,<5','scikit-learn>=1.6,<2',
      'numpy>=2.0,<3','pytest>=8,<9','PyYAML>=6,<7']
subprocess.run([sys.executable,'-m','pip','install','--quiet',*pkgs],check=True)
import torch, braindecode, pyriemann, numpy as np, sklearn, h5py, moabb, mne
versions={k:str(v) for k,v in {'python':sys.version.split()[0], 'torch':torch.__version__,
    'braindecode':braindecode.__version__, 'pyriemann':pyriemann.__version__,
    'numpy':np.__version__, 'sklearn':sklearn.__version__, 'h5py':h5py.__version__,
    'moabb':moabb.__version__, 'mne':mne.__version__}.items()}
print(json.dumps(versions,indent=2)); print('CUDA available:',torch.cuda.is_available())

### 3 · Verify the Phase-2 EEG file (or rebuild only if missing)

Existing private Kaggle Input files are reused **read-only**. Otherwise, the repository's official S3 builder constructs the same complete cache. The training script independently checks the full file's **SHA-256** before opening it for benchmarking.

In [ ]:
EXPECTED_SHA='f6d2c7824f6aaa522286ea8ee5253b0220602117fc69d7b49c3af6e6a1324f7a'
if CACHE_OVERRIDE.strip():
    CACHE=pathlib.Path(CACHE_OVERRIDE).resolve()
    if not CACHE.is_file(): raise FileNotFoundError(CACHE)
else:
    matches = sorted(set(pathlib.Path('/kaggle/input').rglob('physionet_mi_104subjects.h5'))) if pathlib.Path('/kaggle/input').exists() else []
    if len(matches)>1: raise RuntimeError('More than one HDF5 matches; specify CACHE_OVERRIDE.')
    if matches:
        CACHE=matches[0]
    else:
        CACHE=ROOT/'physionet_mi_104subjects.h5'
        print('No Kaggle Input cache found. Build the full cache with official public S3.')
        os.environ.update({'MNE_DATA':str(ROOT/'mne_data_phase3'),
                           'MNE_DATASETS_EEGBCI_PATH':str(ROOT/'mne_data_phase3'),
                           'MOABB_DOWNLOAD_PROVIDER':'upstream'})
        if not CACHE.is_file():
            subprocess.run([sys.executable,'-m','pip','install','--quiet','boto3>=1.34,<2','tqdm>=4.66,<5'],check=True)
            subprocess.run([sys.executable,'-m','scripts.prepare_physionet_s3',
                            '--output',str(CACHE), '--mne-root',os.environ['MNE_DATA'],
                            '--max-new-subjects','104'],check=True)
assert CACHE.is_file(), CACHE
print('HDF5:',CACHE,'bytes:',CACHE.stat().st_size)

### 4 · Materialize reproducible research source and locked split

The following code cell contains the **complete** benchmark runner. It writes the executable source into the pinned checkout, preserving a separate local experimental layer. The script will refuse an incorrect SHA, altered subject split, existing result directory, or any attempt to substitute another cohort.

In [ ]:
runner_source = '"""Phase 3: frozen-cohort, validation-only EEG baselines.\n\nNo held-out test waveforms or test labels are read for model selection.\nModel comparisons and early stopping use validation participants only.\nCPU: Dummy, Riemannian tangent-space regression. GPU: EEGNet, LaBraM probe.\nAll EEG signals stay in the local read-only HDF5, never in artifacts.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport subprocess\nimport time\nimport warnings\nfrom pathlib import Path\n\nimport numpy as np\nfrom sklearn.dummy import DummyClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.metrics import accuracy_score, confusion_matrix, f1_score\n\nfrom src.data import subject_wise_split\nfrom src.portable_eeg import H5SignalSource, inspect_cache, LABELS\nfrom src.subject_statistics import subject_cluster_bootstrap\n\nSHA256_PHASE2 = "f6d2c7824f6aaa522286ea8ee5253b0220602117fc69d7b49c3af6e6a1324f7a"\nDEFAULT_SPLIT = Path("configs/phase3_locked_split.json")\nMODELS = ("dummy", "riemann", "eegnet", "labram_probe")\n\n\ndef sha256(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef validate_frozen_split(y: np.ndarray, subject_ids: np.ndarray, manifest: dict):\n    if len(y) != 18096 or set(np.unique(subject_ids)) != set(\n        i for i in range(1, 110) if i not in (88, 89, 92, 100, 104)\n    ):\n        raise ValueError("Only the 104-subject Phase-2 cohort is allowed.")\n    if manifest["dataset_sha256"] != SHA256_PHASE2 or manifest["split_seed"] != 42:\n        raise ValueError("Frozen dataset hash or split seed unexpectedly changed.")\n    index = subject_wise_split(\n        np.arange(len(y)), y, subject_ids,\n        test_size_subjects=15, val_size_subjects=14, random_state=42,\n    )\n    partition = {}\n    for key, ids in zip(("train", "validation", "test"), index):\n        actual = sorted(map(int, set(subject_ids[ids])))\n        expected = manifest["partitions"][key]\n        if actual != expected["subject_ids"] or len(ids) != expected["epochs_count"]:\n            raise ValueError(f"Frozen {key} split differs from Phase-2 executed manifest.")\n        partition[key] = np.asarray(ids, dtype=np.int64)\n    all_indices = np.concatenate(list(partition.values()))\n    if not np.array_equal(np.sort(all_indices), np.arange(len(y))):\n        raise ValueError("Split partitions are not exhaustive or contain overlap.")\n    return partition\n\n\ndef _features_from_covariances(source, indices, batch_size=32):\n    """Stream only designated participant epochs to OAS covariances."""\n    from pyriemann.estimation import Covariances\n\n    estimator = Covariances(estimator="oas")\n    cov_list = []\n    for offset in range(0, len(indices), batch_size):\n        batch = np.stack([source[int(i)] for i in indices[offset:offset + batch_size]])\n        if not np.all(np.isfinite(batch)):\n            raise ValueError("Non-finite training or validation EEG sample.")\n        cov = estimator.fit_transform(batch)\n        if not np.all(np.isfinite(cov)):\n            raise ValueError("Non-finite OAS covariance matrix.")\n        cov_list.append(np.asarray(cov, dtype=np.float64))\n    return np.concatenate(cov_list)\n\n\ndef _riemann(source, y, train_idx, val_idx, seed, batch_size):\n    from pyriemann.tangentspace import TangentSpace\n\n    t0 = time.monotonic()\n    print("Phase 3 Riemannian: train OAS covariances (streaming)", flush=True)\n    train_cov = _features_from_covariances(source, train_idx, batch_size)\n    print("Phase 3 Riemannian: validation covariances", flush=True)\n    val_cov = _features_from_covariances(source, val_idx, batch_size)\n    tangent = TangentSpace(metric="riemann")\n    X_train = tangent.fit_transform(train_cov)  # training participants ONLY\n    X_val = tangent.transform(val_cov)\n    scaler = StandardScaler()  # MUST fit on training epochs only\n    X_train = scaler.fit_transform(X_train)\n    X_val = scaler.transform(X_val)\n    classifier = LogisticRegression(\n        max_iter=3000, tol=1e-4, class_weight="balanced",\n        random_state=seed, solver="lbfgs",\n    )\n    with warnings.catch_warnings():\n        warnings.filterwarnings("error", category=ConvergenceWarning)\n        classifier.fit(X_train, y[train_idx])\n    if int(np.max(classifier.n_iter_)) >= classifier.max_iter:\n        raise RuntimeError("Riemannian classifier reached iteration limit")\n    if not np.array_equal(classifier.classes_, np.arange(5)):\n        raise ValueError("Incomplete Riemannian class vocabulary.")\n    probs = classifier.predict_proba(X_val)\n    return probs, {"algorithm": "OAS + Riemannian TangentSpace + class-balanced LR",\n                   "tangent_fitted_on_train_only": True, "scaler_fitted_on_train_only": True, "solver": "lbfgs",\n                   "solver_max_iter": classifier.max_iter, "solver_iterations": classifier.n_iter_.tolist(),\n                   "solver_converged": True,\n                   "train_cov_shape": list(train_cov.shape),\n                   "runtime_seconds": round(time.monotonic() - t0, 2)}\n\n\ndef _dummy(y, train_idx, val_idx):\n    model = DummyClassifier(strategy="prior")\n    model.fit(np.zeros((len(train_idx), 1)), y[train_idx])\n    if not np.array_equal(model.classes_, np.arange(5)):\n        raise ValueError("Dummy class vocabulary incomplete.")\n    return model.predict_proba(np.zeros((len(val_idx), 1))), {\n        "strategy": "prior", "training_class_counts": np.bincount(y[train_idx], minlength=5).tolist(),\n        "labels_from_train_only": True,\n    }\n\n\ndef _deep(source, y, train_idx, val_idx, seed, *, model_name, epochs, patience,\n          batch_size, lr, weight_decay, device, checkpoint_path):\n    import torch\n    from torch import nn\n    from torch.utils.data import Dataset, DataLoader\n    from sklearn.utils.class_weight import compute_class_weight\n\n    class EpochData(Dataset):\n        def __init__(self, rows):\n            self.rows = np.asarray(rows, dtype=np.int64)\n\n        def __len__(self):\n            return len(self.rows)\n\n        def __getitem__(self, index):\n            row = int(self.rows[index])\n            return (torch.from_numpy(np.array(source[row], dtype=np.float32, copy=True)),\n                    torch.tensor(int(y[row]), dtype=torch.long))\n\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    from braindecode.models import EEGNet\n    if model_name == "eegnet":\n        model = EEGNet(n_chans=64, n_outputs=5, n_times=400, sfreq=160.)\n        configuration = {"architecture": "braindecode.EEGNet", "pretrained": False}\n    elif model_name == "labram_probe":\n        from src.model import load_labram_model\n        from src.training_modes import configure_trainable_parameters\n        model = load_labram_model(\n            n_outputs=5, n_times=400, n_chans=64, sfreq=160.,\n            pretrained_repo_id="braindecode/labram-pretrained",\n            pretrained_revision="0563b6c626e7b40d9a36653b763715db94d945d7",\n        )\n        configuration = {\n            "architecture": "LaBraM 400-sample pretrained prefix adapter",\n            "pretrained": True,\n            "pretrained_revision": "0563b6c626e7b40d9a36653b763715db94d945d7",\n            "parameter_audit": configure_trainable_parameters(\n                model, strategy="linear_probe", n_outputs=5,\n            ),\n        }\n    else:\n        raise ValueError(model_name)\n    model = model.to(device)\n    tr = DataLoader(EpochData(train_idx), batch_size=batch_size, shuffle=True, num_workers=0)\n    va = DataLoader(EpochData(val_idx), batch_size=batch_size, shuffle=False, num_workers=0)\n    weights = compute_class_weight("balanced", classes=np.arange(5), y=y[train_idx])\n    criterion = nn.CrossEntropyLoss(weight=torch.as_tensor(weights, device=device, dtype=torch.float32))\n    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad),\n                                  lr=lr, weight_decay=weight_decay)\n    history = []\n    best_f1 = -1.0\n    best_probs = None\n    best_weights = None\n    best_epoch = 0\n    stale = 0\n    for epoch in range(1, epochs + 1):\n        if model_name == "labram_probe":\n            model.eval()  # frozen feature extractor must not change dropout/BN state\n            model.get_submodule("final_layer").train()\n        else:\n            model.train()\n        total_loss = 0.\n        for xb, yb in tr:\n            xb, yb = xb.to(device), yb.to(device)\n            optimizer.zero_grad(set_to_none=True)\n            logits = model(xb)\n            loss = criterion(logits, yb)\n            if not torch.isfinite(loss):\n                raise RuntimeError(f"Nonfinite {model_name} training loss")\n            loss.backward()\n            optimizer.step()\n            total_loss += float(loss.detach().cpu().item()) * len(xb)\n        model.eval()\n        probabilities = []\n        with torch.no_grad():\n            for xb, _ in va:\n                logits = model(xb.to(device))\n                probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())\n        probs = np.concatenate(probabilities)\n        metric = score_validation(y[val_idx], probs)["macro_f1"]\n        history.append({"epoch": epoch, "train_loss": total_loss / len(tr.dataset),\n                        "validation_macro_f1": metric})\n        print(f"{model_name} epoch={epoch} train_loss={history[-1][\'train_loss\']:.5f} "\n              f"validation_macro_f1={metric:.5f}", flush=True)\n        if metric > best_f1 + 1e-12:\n            best_f1 = metric\n            best_epoch = epoch\n            best_probs = probs.copy()\n            best_weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n            stale = 0\n        else:\n            stale += 1\n        if stale >= patience:\n            break\n    if best_probs is None or best_weights is None:\n        raise RuntimeError(f"No valid {model_name} validation checkpoint.")\n    checkpoint_path = Path(checkpoint_path)\n    if checkpoint_path.exists():\n        raise FileExistsError("Refusing to overwrite selected model weights")\n    checkpoint_path.parent.mkdir(parents=True, exist_ok=False)\n    import torch\n    torch.save(best_weights, checkpoint_path)\n    restored = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n    model.load_state_dict(restored, strict=True)\n    model.eval()\n    verified_probs = []\n    with torch.inference_mode():\n        for xb, _ in va:\n            verified_probs.append(torch.softmax(model(xb.to(device)), dim=1).cpu().numpy())\n    verified_probs = np.concatenate(verified_probs)\n    if not np.allclose(verified_probs, best_probs, rtol=1e-5, atol=1e-6):\n        raise RuntimeError("Persisted best model predictions mismatch selected validation epoch")\n    checkpoint_digest = sha256(checkpoint_path)\n    # No test predictions, even after validation-only checkpoint selection.\n    configuration.update({"device": str(device), "best_epoch": best_epoch,\n                          "epochs_completed": len(history), "history": history,\n                          "learning_rate": lr, "weight_decay": weight_decay,\n                          "class_weights": weights.tolist(),\n                          "best_state_dict_saved": True,\n                          "selected_checkpoint_sha256": checkpoint_digest,\n                          "selected_checkpoint_bytes": checkpoint_path.stat().st_size,\n                          "checkpoint_reload_validation_predictions_match": True})\n    return best_probs, configuration\n\n\ndef score_validation(y_true: np.ndarray, y_prob: np.ndarray) -> dict:\n    if y_prob.shape != (len(y_true), 5) or not np.all(np.isfinite(y_prob)):\n        raise ValueError("Invalid five-class validation predictions")\n    if np.any(y_prob < 0) or not np.allclose(y_prob.sum(1), 1., atol=1e-5):\n        raise ValueError("Invalid probability rows")\n    if set(y_true) != set(range(5)):\n        raise ValueError("All five classes must occur in validation.")\n    predictions = y_prob.argmax(1)\n    recalls = [(predictions[y_true == i] == i).mean() for i in range(5)]\n    return {\n        "accuracy": float(accuracy_score(y_true, predictions)),\n        "macro_f1": float(f1_score(y_true, predictions, labels=np.arange(5),\n                                   average="macro", zero_division=0)),\n        "balanced_accuracy": float(np.mean(recalls)),\n        "per_class_recall": [float(x) for x in recalls],\n        "confusion_matrix": confusion_matrix(y_true, predictions,\n                                               labels=np.arange(5)).tolist(),\n    }\n\n\ndef _dataset_manifest(path, data_sha, args, partitions, y, subjects) -> dict:\n    try:\n        commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()\n    except (OSError, subprocess.CalledProcessError):\n        commit = "unknown: source supplied independently"\n    return {\n        "source_commit": commit,\n        "cache_basename": path.name,\n        "cache_sha256": data_sha,\n        "split_seed": 42,\n        "random_seed": args.seed,\n        "subjects": {key: sorted(map(int, set(subjects[rows]))) for key, rows in partitions.items()},\n        "epoch_counts": {key: int(len(rows)) for key, rows in partitions.items()},\n        "class_names": list(LABELS),\n        "preprocessing": "PhysioNet EEGMMIDB 1.0.0 / MOABB 1.7.2 / 8-32Hz / 160Hz / uV / first 400 samples",\n        "validation_class_counts": np.bincount(y[partitions["validation"]], minlength=5).tolist(),\n        "python": platform.python_version(),\n        "numpy": np.__version__,\n        "device": args.device,\n        "evaluation_scope": "validation_only_no_heldout_test_labels_used",\n    }\n\n\ndef run(args):\n    path = Path(args.cache).resolve()\n    if not path.is_file():\n        raise FileNotFoundError(f"Attach the validated Phase 2 HDF5 to Kaggle: {path}")\n    if args.seed < 0 or args.batch_size < 1 or args.epochs < 1 or args.patience < 1:\n        raise ValueError("Invalid training configuration")\n    models = args.models\n    if not models or any(m not in MODELS for m in models) or len(set(models)) != len(models):\n        raise ValueError("Models must be a unique non-empty subset of known baselines.")\n    if args.output_dir.exists():\n        raise FileExistsError(f"Refusing to overwrite Phase 3 evidence: {args.output_dir}")\n    data_sha = sha256(path)\n    if data_sha != SHA256_PHASE2:\n        raise ValueError("Refusing modified or unverifiable Phase 2 EEG HDF5 checksum.")\n    labels, subjects, shape = inspect_cache(path, require_complete=True)\n    if tuple(shape) != (18096, 64, 400):\n        raise ValueError(f"HDF5 schema shape changed: {shape}")\n    if set(np.unique(labels)) != set(LABELS):\n        raise ValueError("Unexpected label classes or order.")\n    vocabulary = {label: index for index, label in enumerate(LABELS)}\n    y = np.fromiter((vocabulary[label] for label in labels), dtype=np.int64, count=len(labels))\n    manifest = json.loads(Path(args.split_manifest).read_text(encoding="utf-8"))\n    partitions = validate_frozen_split(y, subjects, manifest)\n    args.output_dir.mkdir(parents=True, exist_ok=False)\n    root = _dataset_manifest(path, data_sha, args, partitions, y, subjects)\n    root["split_manifest_sha256"] = sha256(Path(args.split_manifest))\n    root["models_requested"] = models\n    (args.output_dir / "protocol.json").write_text(json.dumps(root, indent=2) + "\\n")\n    src = H5SignalSource(path, shape)\n    try:\n        for model in models:\n            start = time.monotonic()\n            print(f"[Phase 3] RUN {model}; validation-only; test locked", flush=True)\n            if model == "dummy":\n                probabilities, hyper = _dummy(y, partitions["train"], partitions["validation"])\n            elif model == "riemann":\n                probabilities, hyper = _riemann(src, y, partitions["train"],\n                                                 partitions["validation"], args.seed,\n                                                 args.batch_size)\n            else:\n                probabilities, hyper = _deep(src, y, partitions["train"],\n                                            partitions["validation"], args.seed,\n                                            model_name=model, epochs=args.epochs,\n                                            patience=args.patience, batch_size=args.batch_size,\n                                            lr=args.learning_rate, weight_decay=args.weight_decay,\n                                            device=args.device,\n                                            checkpoint_path=args.output_dir / model / "best_weights.pt")\n            metrics = score_validation(y[partitions["validation"]], probabilities)\n            stats = subject_cluster_bootstrap(\n                y[partitions["validation"]], probabilities,\n                subjects[partitions["validation"]], n_bootstraps=args.bootstrap,\n                seed=args.seed,\n            )\n            report = {\n                "status": "completed", "model": model,\n                "validation_metrics": metrics,\n                "validation_subject_cluster_intervals": stats,\n                "configuration": hyper,\n                "elapsed_seconds": round(time.monotonic() - start, 1),\n                "no_test_predictions_computed": True,\n                "protocol_sha256": sha256(args.output_dir / "protocol.json"),\n            }\n            target = args.output_dir / model\n            target.mkdir(exist_ok=True)\n            (target / "report.json").write_text(json.dumps(report, indent=2) + "\\n")\n            np.savez_compressed(target / "validation_predictions.npz",\n                                y_true=y[partitions["validation"]],\n                                y_prob=probabilities,\n                                subject_ids=subjects[partitions["validation"]],\n                                class_names=np.asarray(LABELS))\n            print(f"[Phase 3] {model} val Macro-F1={metrics[\'macro_f1\']:.4f}; "\n                  f"BalancedAcc={metrics[\'balanced_accuracy\']:.4f}", flush=True)\n    finally:\n        src.close()\n    existing = sorted(p.name for p in args.output_dir.iterdir() if p.is_dir())\n    all_done = sorted(models) == existing\n    summary = {"status": "completed" if all_done else "partial",\n               "models_completed": existing, "models_requested": models,\n               "evaluation_scope": "validation_only", "test_evaluations": 0,\n               "dataset_sha256": data_sha}\n    (args.output_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\\n")\n    if not all_done:\n        raise RuntimeError("Missing a requested model.")\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--cache", required=True)\n    parser.add_argument("--split-manifest", type=Path, default=DEFAULT_SPLIT)\n    parser.add_argument("--output-dir", type=Path, default=Path("artifacts/phase3_validation"))\n    parser.add_argument("--models", nargs="+", choices=MODELS, default=list(MODELS))\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--epochs", type=int, default=30)\n    parser.add_argument("--patience", type=int, default=6)\n    parser.add_argument("--batch-size", type=int, default=32)\n    parser.add_argument("--learning-rate", type=float, default=1e-3)\n    parser.add_argument("--weight-decay", type=float, default=1e-4)\n    parser.add_argument("--bootstrap", type=int, default=250)\n    parser.add_argument("--device", default="cuda" if __import__("importlib").util.find_spec("torch") and __import__("torch").cuda.is_available() else "cpu")\n    args = parser.parse_args()\n    if args.bootstrap < 10 or args.device not in ("cuda", "cpu"):\n        parser.error("Bootstrap must be >=10 and device cpu or cuda")\n    result = run(args)\n    print(json.dumps(result, indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
locked_split_json = '{\n  "split_seed": 42,\n  "dataset_sha256": "f6d2c7824f6aaa522286ea8ee5253b0220602117fc69d7b49c3af6e6a1324f7a",\n  "partitions": {\n    "test": {\n      "subject_ids": [\n        1,\n        11,\n        19,\n        31,\n        32,\n        46,\n        48,\n        54,\n        65,\n        66,\n        78,\n        91,\n        95,\n        97,\n        99\n      ],\n      "epochs_count": 2610\n    },\n    "train": {\n      "subject_ids": [\n        3,\n        4,\n        5,\n        7,\n        8,\n        9,\n        10,\n        12,\n        14,\n        16,\n        17,\n        18,\n        20,\n        21,\n        23,\n        24,\n        25,\n        27,\n        28,\n        29,\n        30,\n        33,\n        34,\n        35,\n        37,\n        38,\n        40,\n        41,\n        42,\n        43,\n        44,\n        45,\n        47,\n        49,\n        51,\n        53,\n        55,\n        56,\n        57,\n        59,\n        60,\n        61,\n        63,\n        64,\n        67,\n        68,\n        69,\n        70,\n        71,\n        72,\n        73,\n        74,\n        76,\n        77,\n        79,\n        80,\n        81,\n        82,\n        83,\n        84,\n        85,\n        86,\n        87,\n        90,\n        93,\n        94,\n        96,\n        101,\n        102,\n        103,\n        105,\n        106,\n        107,\n        108,\n        109\n      ],\n      "epochs_count": 13050\n    },\n    "validation": {\n      "subject_ids": [\n        2,\n        6,\n        13,\n        15,\n        22,\n        26,\n        36,\n        39,\n        50,\n        52,\n        58,\n        62,\n        75,\n        98\n      ],\n      "epochs_count": 2436\n    }\n  }\n}\n'
test_source = '"""Deterministic validation-only and split-lock negative controls."""\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pytest\n\nfrom scripts.run_phase3_validation import (\n    SHA256_PHASE2,\n    _dummy,\n    score_validation,\n    validate_frozen_split,\n)\n\nLOCK = Path(__file__).resolve().parents[1] / "configs/phase3_locked_split.json"\n\n\ndef test_dummy_prior_never_uses_validation_labels():\n    y = np.asarray([3] * 10 + [0, 1, 2, 4] + [0, 1, 2, 3, 4])\n    train = np.arange(14)\n    valid = np.arange(14, 19)\n    prob, meta = _dummy(y, train, valid)\n    assert prob.shape == (5, 5)\n    assert np.allclose(prob[:, 3], 10 / 14)\n    assert meta["labels_from_train_only"]\n    score = score_validation(y[valid], prob)\n    assert score["balanced_accuracy"] == pytest.approx(0.2)\n\n\ndef test_prediction_metrics_reject_nonprobabilities():\n    y = np.arange(5)\n    prob = np.eye(5)\n    assert score_validation(y, prob)["macro_f1"] == 1.0\n    prob[0, :] = 0\n    with pytest.raises(ValueError, match="probability"):\n        score_validation(y, prob)\n    with pytest.raises(ValueError, match="five-class"):\n        score_validation(y, np.ones((5, 4)) / 4)\n\n\ndef test_frozen_split_exactly_matches_phase2_subject_ids():\n    lock = json.loads(LOCK.read_text())\n    counts = lock["partitions"]\n    assert [len(counts[k]["subject_ids"]) for k in ("train", "validation", "test")] == [75, 14, 15]\n    assert [counts[k]["epochs_count"] for k in ("train", "validation", "test")] == [13050, 2436, 2610]\n    assert lock["dataset_sha256"] == SHA256_PHASE2\n    parts = [set(counts[k]["subject_ids"]) for k in ("train", "validation", "test")]\n    assert set.union(*parts) == (set(range(1, 110)) - {88, 89, 92, 100, 104})\n    assert not any(parts[i] & parts[j] for i in range(3) for j in range(i + 1, 3))\n\n\ndef test_tampered_split_fails_closed():\n    lock = json.loads(LOCK.read_text())\n    all_subjects = np.concatenate([\n        np.repeat(s, 174) for s in sorted(set(range(1, 110)) - {88, 89, 92, 100, 104})\n    ])\n    # The real dataset has exactly 174 epochs per retained subject.\n    assert len(all_subjects) == 18096\n    y = np.zeros(len(all_subjects), dtype=np.int64)\n    groups = validate_frozen_split(y, all_subjects, lock)\n    assert len(groups["test"]) == 2610\n    lock["partitions"]["validation"]["subject_ids"][0] = 1\n    with pytest.raises(ValueError, match="Frozen validation split"):\n        validate_frozen_split(y, all_subjects, lock)\n'
(WORKSPACE/'scripts'/'run_phase3_validation.py').write_text(runner_source,encoding='utf-8')
(WORKSPACE/'configs'/'phase3_locked_split.json').write_text(locked_split_json,encoding='utf-8')
(WORKSPACE/'tests'/'test_phase3_validation.py').write_text(test_source,encoding='utf-8')
source_sha=hashlib.sha256(runner_source.encode('utf-8')).hexdigest()
print('Runner source SHA-256:',source_sha)
subprocess.run([sys.executable,'-m','compileall','-q','scripts/run_phase3_validation.py'],check=True)
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_phase3_validation.py'],check=True)

### 5 · Run all four validation-only benchmarks

Riemannian preprocessing and classification are fitted only on training examples. EEGNet and LaBraM use **validation Macro-F1** for early stopping. LaBraM uses its pinned pretrained backbone and trains the head only. The script saves validation predictions and per-subject uncertainty; it **never reads test waveforms** or produces test predictions. If any step fails, this notebook still packages the logs and partial data so errors can be diagnosed honestly.

In [ ]:
import traceback
if OUTPUT.exists() or RESULT_ZIP.exists():
    raise FileExistsError('Existing output exists. Rename/delete manually before a fresh independent run.')
command=[sys.executable,'-m','scripts.run_phase3_validation',
         '--cache',str(CACHE), '--output-dir',str(OUTPUT),
         '--models',*MODELS, '--seed',str(SEED), '--epochs',str(EPOCHS),
         '--patience',str(PATIENCE), '--batch-size',str(BATCH_SIZE),
         '--bootstrap',str(BOOTSTRAPS),
         '--device','cuda' if torch.cuda.is_available() else 'cpu']
print('EXEC:', ' '.join(command[:8]),'...',flush=True)
RUN_OK=False
log_path=ROOT/'phase3_rerun_execution.log'
with open(log_path,'w',encoding='utf-8') as log:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,bufsize=1,cwd=WORKSPACE)
    for line in process.stdout:
        log.write(line);log.flush()
        print(line,end='',flush=True)
    code=process.wait()
    RUN_OK=(code==0)
print('Phase 3 command result:', 'SUCCEEDED' if RUN_OK else f'FAILED (return code {code})')

### 6 · Package **results only**, not the EEG dataset

Download the ZIP even if an experiment failed. It includes logs, protocol metadata, all completed validation metrics, and code checksums. **No raw HDF5, EDF, model weights, or test-set predictions** are included.

In [ ]:
package_manifest={'phase':'3','status':'completed' if RUN_OK else 'incomplete',
                  'code_revision':revision,'runner_sha256':source_sha,
                  'expected_cache_sha256':EXPECTED_SHA,'device':str('cuda' if torch.cuda.is_available() else 'cpu'),
                  'models_requested':MODELS,'versions':versions,
                  'warning':'All model metrics in this package are VALIDATION-only, not confirmatory test scores.'}
manifest_path=ROOT/'phase3_rerun_packaging_manifest.json'
manifest_path.write_text(json.dumps(package_manifest,indent=2)+'\n')
with zipfile.ZipFile(RESULT_ZIP,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    z.write(log_path,arcname='phase3_execution.log')
    z.write(manifest_path,arcname='phase3_packaging_manifest.json')
    z.writestr('source_sha256.txt',source_sha+'\n')
    z.writestr('locked_split_manifest.json',locked_split_json)
    if OUTPUT.exists():
        for f in sorted(OUTPUT.rglob('*')):
            if f.is_file() and f.suffix in {'.json','.npz','.csv','.txt'}:
                z.write(f,arcname='results/'+str(f.relative_to(OUTPUT)))
print('Results ZIP:',RESULT_ZIP, 'size:',RESULT_ZIP.stat().st_size)
if not RUN_OK:
    print('AT LEAST ONE MODEL FAILED. Share ZIP for diagnosis; Phase 3 remains blocked.')
else:
    summaries=[]
    for name in MODELS:
        path=OUTPUT/name/'report.json'
        if path.exists():
            j=json.loads(path.read_text());m=j['validation_metrics']
            summaries.append({'model':name,'validation_macro_f1':m['macro_f1'],
                'validation_balanced_accuracy':m['balanced_accuracy'],
                'validation_accuracy':m['accuracy']})
    print(json.dumps(summaries,indent=2))
    print('Full model comparison is for validation-only; a scientific review is still required.')